Here I'll describe the acquisition of the Human Coronavirus background.

- The sequences were downloaded from the NCBI viruses on 09.Sep.2026.
- No initial filters were applyed.


In [5]:
import pandas as pd

Here I'll create a function to filter and prepare the background metadata for the downstream analysis.
This function will: 
- Create a new header for the sequences
    - This reader will be free of spaces
    - And also will contain this structure Accession|Country|Collection_Date
- Keep only genomes with Country and Collection Date (Year at least)
- Make the date standard for Augur (Y-MM-DD or Y-MM-XX or Y-XX-XX)
- Filter the genomes by lenth (>=80%)
- Filter out other hosts than "Homo sapiens"

Now I'll perform a standardisation of the dates on the dataframe to fit into the Augur framework.

## Setting the functions
1. Function to standardise the format in Date column
- since the Augur pipeleni deals with uncertanty with -XX or -XX-XX, I'll make my dates to fit in this format.
 

In [6]:
from datetime import datetime

#This function test if the date is in the format
def is_date_format(text, date_format):
    try:
        datetime.strptime(text, date_format)
        return True
    except ValueError:
        return False


#this function will test each different date format until find the right one
def classify_date(test_str):

    if pd.isna(test_str):
        return None

    if is_date_format(test_str, "%Y-%m-%d"):
        return test_str

    elif is_date_format(test_str, "%Y-%m"):
        return test_str+"-XX"

    elif is_date_format(test_str, "%Y"):
        return test_str+"-XX-XX"

    else:
        return "error"
    

2. Function to run the date format and to filter the data.
- I'll put everything into a single function to decrease the amount of lines in the code.

In [ ]:
def filter_data(df, ref_length, min_coverage=0.8):

    # Running the date format function
    df["Date"] = df["Collection_Date"].apply(classify_date)
    
    # creating a new column for the head
    # it will be the paste values from Accession | Country | Collection_date
    df["name"] = (
        df["Accession"].astype(str) + "|" +
        df["Country"].astype(str) + "|" +
        df["Date"].astype(str)
    )    

    # Replacing space for _
    df["name"] = df["name"].str.replace(" ", "_")
    
    # Calculating coverage
    df["coverage"] = df["Length"]/ref_length    

    # keeping only lines with country or date 
    # and coverage >= 80%
    df_filt = (
        df[df["coverage"] >= min_coverage]
        .dropna(subset=["Country","Collection_Date"])
        .query("Host == 'Homo sapiens'")
    )

    
    return df_filt


In [8]:
# how to run the funtion line by line.
# its now incorporated into the function

#metadata_229e_filt["Date"] = metadata_229e_filt["Collection_Date"].apply(classify_date)


- HCoV 229e

In [18]:
# Importing the metadata df for 229E
metadata_229e = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_metadata_09Sep26.csv")

metadata_229e_filt = filter_data(df= metadata_229e, 
                                ref_length= 27317, 
                                min_coverage= 0.8
                                )

metadata_229e_filt.shape

metadata_229e_filt.shape


(549, 30)

- HCoV HKU1

In [19]:
# Importing the metadata df for HKU1
metadata_HKU1 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/HKU1/NCBI_virus/HKU1_metadata_09Sep26.csv")

metadata_HKU1_filt = filter_data(df= metadata_HKU1, 
                                ref_length= 29926, 
                                min_coverage= 0.8
                                )

metadata_HKU1_filt.shape

(396, 30)

- HCoV NL63

In [20]:
# Importing the metadata df for NL63
metadata_NL63 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/NL63/NCBI_virus/NL63_metadata_09Sep26.csv")

metadata_NL63_filt = filter_data(df= metadata_NL63, 
                                ref_length= 27553, 
                                min_coverage= 0.8
                                )

metadata_NL63_filt.shape

(417, 30)

- HCoV OC43

In [22]:
# Importing the metadata df for OC43
metadata_OC43 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/OC43/NCBI_virus/OC43_metadata_09Sep26.csv")

metadata_OC43_filt = filter_data(df= metadata_OC43, 
                                ref_length= 30741, 
                                min_coverage= 0.8
                                )

metadata_OC43_filt.shape

(844, 30)

## Genotypes
Now I'll Load a new metadata with the genotypes for each HCoV species.

In [ ]:
from pandas import merge
genotyes_229E = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/229E/HCoV_229E_genotypes.csv")


In [49]:
metadata_229e_filt["Accession_clean"] = (
    metadata_229e_filt["Accession"]
    .str.split(".")
    .str[0]
)

metadata_229e_final = metadata_229e_filt.merge(
    genotyes_229E,
    left_on="Accession_clean",
    right_on="accession_or_isolate",
    how="inner"
)

metadata_229e_final = pd.concat([metadata_229e_final, metadata_229e_filt])

metadata_229e_final = metadata_229e_final.drop_duplicates(subset=["Accession"])